# Notebook 08 — Full Multimodal Pipeline (Text + Figures → Grounded Answer)

**Phase 5 · Task group 118.** Goal: stitch together everything the previous notebooks built into one end-to-end pipeline — query → hybrid text retrieval → cross-encoder rerank → CLIP figure retrieval → **late fusion** → LLM answer with citations.

### The late-fusion recipe
1. Run hybrid retrieval + cross-encoder rerank over the **text** Chroma collection → top-5 text passages.
2. Run CLIP text→image search over the **figures** Chroma collection → top-3 figures.
3. Merge both evidence lists into a single prompt. Text gets `[T1] … [T5]` tags, figures get `[F1] … [F3]` tags plus their captions.
4. Call Gemini (or a local LLM) with a tightly constrained grounding prompt. Require it to cite tag IDs.

### Generator options
* **Gemini 1.5 Flash** via Google AI Studio — free-tier friendly, fast, good at grounding.
* **Local Mistral 7B** — fully offline, slower; kept as a fallback (turned off by default).
* **LLaVA-1.5-7B** — VL model used only when the router decides the query is figure-centric.

### What this ships
1. All components loaded and counts printed.
2. Late fusion retrieval test — confirm both text and figure hits come back.
3. Full answer generation test — verify the answer cites retrieved tags, no hallucinated paper titles.
4. `multimodal_config_<tier>.json` saved so the Gradio demo (`5_app/`) can rehydrate without rebuilding indexes.


In [ ]:
import os, sys, json, pickle, time, warnings
from pathlib import Path
from collections import defaultdict

NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if (NB_DIR.parent / "2_src").exists() else NB_DIR
sys.path.insert(0, str(PROJECT_ROOT / "2_src"))

os.environ.setdefault("SCIRET_TIER", "tier1")
from config import (
    get_config, BGE_M3_MODEL, CROSS_ENCODER_MODEL, CLIP_MODEL, CLIP_DIM,
    GEMINI_MODEL, DENSE_TOP_K, SPARSE_TOP_K, RRF_K, RERANK_TOP_K, SEED,
)
CFG = get_config()
print(CFG.summary())

import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")


## 1. Load every component

In [ ]:
import torch, chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import CLIPModel, CLIPProcessor
from rank_bm25 import BM25Okapi

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)

bge = SentenceTransformer(BGE_M3_MODEL, device=DEVICE)
rer = CrossEncoder(CROSS_ENCODER_MODEL, device=DEVICE)
clip = CLIPModel.from_pretrained(CLIP_MODEL).to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained(CLIP_MODEL)

client = chromadb.PersistentClient(path=str(CFG.chroma_dir))
text_col = client.get_collection(CFG.text_collection)
try:
    fig_col = client.get_collection(CFG.figure_collection)
except Exception:
    fig_col = None
    print("figures collection missing — text-only mode")

bm25, bm25_ids = pickle.loads((CFG.embeddings_dir / "bm25_index.pkl").read_bytes())

chunks = pd.read_parquet(CFG.chunks_path)
text_lookup  = dict(zip(chunks["chunk_id"], chunks["chunk_text"]))
title_lookup = dict(zip(chunks["chunk_id"], chunks["title"]))
cord_lookup  = dict(zip(chunks["chunk_id"], chunks["cord_uid"]))

print("text collection:", text_col.count())
print("fig collection :", fig_col.count() if fig_col else 0)
print("BM25 docs      :", len(bm25_ids))
print("chunks loaded  :", len(chunks))


## 2. Retrieval primitives

In [ ]:
def embed_clip_text(texts):
    with torch.no_grad():
        inp = clip_proc(text=texts, return_tensors="pt", padding=True, truncation=True, max_length=77).to(DEVICE)
        v = clip.get_text_features(**inp).cpu().numpy()
    v = v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-9)
    return v.astype("float32")

def bm25_query(q, k=SPARSE_TOP_K):
    scores = bm25.get_scores([w for w in q.lower().split() if w.strip()])
    top = np.argsort(-scores)[:k]
    return [(bm25_ids[int(i)], float(scores[int(i)])) for i in top]

def dense_query(q, k=DENSE_TOP_K):
    qv = bge.encode([q], normalize_embeddings=True)[0].tolist()
    r = text_col.query(query_embeddings=[qv], n_results=k, include=["distances"])
    return list(zip(r["ids"][0], [1-float(d) for d in r["distances"][0]]))

def rrf(runs, k=RRF_K, top_k=100):
    s = defaultdict(float)
    for run in runs:
        for rank,(d,_) in enumerate(run, 1):
            s[d] += 1.0 / (k + rank)
    return sorted(s.items(), key=lambda x: x[1], reverse=True)[:top_k]

def text_retrieve(q, candidate_k=100, top_k=RERANK_TOP_K):
    cand = rrf([dense_query(q), bm25_query(q)], top_k=candidate_k)
    ids = [d for d,_ in cand]
    ce = rer.predict([[q, text_lookup.get(i,"")] for i in ids], show_progress_bar=False)
    order = np.argsort(-ce)[:top_k]
    return [{"chunk_id": ids[int(i)], "ce_score": float(ce[int(i)]),
             "title": title_lookup.get(ids[int(i)],""),
             "text": text_lookup.get(ids[int(i)],""),
             "cord_uid": cord_lookup.get(ids[int(i)],"")} for i in order]

def figure_retrieve(q, top_k=3, min_sim=0.18):
    if fig_col is None or fig_col.count() == 0:
        return []
    qv = embed_clip_text([q])[0].tolist()
    r = fig_col.query(query_embeddings=[qv], n_results=top_k,
                      include=["distances","documents","metadatas"])
    out = []
    for i,d,doc,m in zip(r["ids"][0], r["distances"][0], r["documents"][0], r["metadatas"][0]):
        sim = 1.0 - float(d)
        if sim < min_sim: continue
        out.append({"figure_id": i, "sim": round(sim,3),
                    "caption": doc, "asset_path": m.get("asset_path",""),
                    "cord_uid": m.get("cord_uid","")})
    return out

print("retrieval primitives ready")


## 3. Late fusion test

In [ ]:
DEMO = "What imaging techniques were used to study COVID-19 lung damage?"

txt_hits = text_retrieve(DEMO)
fig_hits = figure_retrieve(DEMO)

print("TEXT HITS")
for i,h in enumerate(txt_hits,1):
    print(f"  [T{i}] ({h['ce_score']:.2f}) {h['title'][:80]}")
    print(f"        {h['text'][:150]}...")
print("FIG HITS")
for i,h in enumerate(fig_hits,1):
    print(f"  [F{i}] ({h['sim']:.2f}) {h['figure_id']}  | {h['caption'][:100]}")


## 4. Prompt builder + generator

We keep the prompt deliberately strict: every factual claim must cite a tag. Unsupported claims must be marked as such. This matters for RQ4 (hallucination).

In [ ]:
SYSTEM = (
    "You are a scientific assistant answering questions about COVID-19 "
    "using only the evidence provided in EVIDENCE. Every factual claim MUST "
    "cite one or more tags in square brackets, e.g. [T1], [F2]. "
    "If the evidence is insufficient, say so explicitly. "
    "Do NOT invent paper titles, author names, or statistics that are not in EVIDENCE."
)

def build_prompt(query, txt_hits, fig_hits):
    evid = []
    for i,h in enumerate(txt_hits, 1):
        evid.append(f"[T{i}] title='{h['title']}'  passage='{h['text'][:600]}'")
    for i,h in enumerate(fig_hits, 1):
        evid.append(f"[F{i}] figure_id='{h['figure_id']}'  caption='{h['caption'][:300]}'")
    body = "\n".join(evid) if evid else "(no evidence retrieved)"
    return f"""SYSTEM: {SYSTEM}

EVIDENCE:
{body}

QUESTION: {query}

INSTRUCTIONS:
- Answer in 4-8 sentences.
- Every claim must cite a tag like [T1] or [F2].
- If evidence is weak or missing, say 'evidence is insufficient'.
ANSWER:"""


In [ ]:
# Generator abstraction: Gemini if a key is available, else local fallback.
GEMINI_KEY = os.environ.get("GEMINI_API_KEY")

def generate_answer(query, txt_hits, fig_hits):
    prompt = build_prompt(query, txt_hits, fig_hits)
    if GEMINI_KEY:
        try:
            import google.generativeai as genai
            genai.configure(api_key=GEMINI_KEY)
            m = genai.GenerativeModel(GEMINI_MODEL)
            r = m.generate_content(prompt, generation_config={"temperature":0.2, "max_output_tokens":400})
            return r.text
        except Exception as e:
            print("gemini failed, falling back:", e)
    # local fallback — deterministic composition
    tags = ", ".join([f"[T{i}]" for i in range(1, len(txt_hits)+1)] +
                     [f"[F{i}]" for i in range(1, len(fig_hits)+1)])
    if not txt_hits and not fig_hits:
        return "Evidence is insufficient — no chunks or figures were retrieved for this query."
    preview = " ".join((txt_hits[0]['text'] if txt_hits else '').split()[:60])
    return (f"Based on the retrieved evidence {tags}, {preview}. "
            "Additional supporting material appears in the remaining tagged passages. "
            "Note: this response was generated locally without an LLM; set GEMINI_API_KEY for graded answers.")

answer = generate_answer(DEMO, txt_hits, fig_hits)
print("ANSWER:\n", answer)


## 5. Hallucination sanity check

We pull every paper title from the retrieved evidence and assert that any title the answer mentions is actually present in the evidence. This is a lightweight proxy for RQ4.

In [ ]:
def titles_in(text):
    """return evidence titles that the answer quotes verbatim"""
    return [t for t in set(title_lookup.values()) if t and t.lower() in text.lower()]

evidence_titles = set(h["title"] for h in txt_hits if h["title"])
mentioned = set(titles_in(answer))
hallucinated = mentioned - evidence_titles
print("titles in answer     :", len(mentioned))
print("titles in evidence   :", len(evidence_titles))
print("possibly hallucinated:", len(hallucinated))
for t in list(hallucinated)[:3]:
    print(" -", t)


## 6. Persist `multimodal_config_<tier>.json`

Everything a downstream process (Gradio app, eval notebook) needs to rehydrate the pipeline without rebuilding: tier, collection names, model ids, retrieval hyperparameters.

In [ ]:
cfg_out = {
    "tier": CFG.tier,
    "project_root": str(CFG.project_root),
    "chroma_dir": str(CFG.chroma_dir),
    "text_collection": CFG.text_collection,
    "figure_collection": CFG.figure_collection,
    "bm25_index": str(CFG.embeddings_dir / "bm25_index.pkl"),
    "chunks_parquet": str(CFG.chunks_path),
    "figures_metadata_parquet": str(CFG.figures_metadata_path),
    "models": {
        "text_embedder":  BGE_M3_MODEL,
        "reranker":       CROSS_ENCODER_MODEL,
        "vision":         CLIP_MODEL,
        "generator":      GEMINI_MODEL,
    },
    "retrieval": {
        "dense_top_k":  DENSE_TOP_K,
        "sparse_top_k": SPARSE_TOP_K,
        "rrf_k":        RRF_K,
        "rerank_top_k": RERANK_TOP_K,
        "figure_top_k": 3,
        "figure_min_sim": 0.18,
    },
    "seed": SEED,
}
out_path = CFG.multimodal_config_path
out_path.write_text(json.dumps(cfg_out, indent=2))
print("saved ->", out_path)


---
**Outputs**
* `2_src/multimodal_config_<tier>.json` — pipeline rehydration manifest
* Evidence that late fusion + grounded generation work end-to-end.

**Next:** Notebook 09 — RAGAS evaluation across SciRet 2022 / v2 text / v2 multimodal.
